# C 단계 실용 변형 생성: FT5 + PRUNE50 + PTQ_INT8 + PRUNE20

이 노트북은 지정된 A parent 10개에서 **Fine-tuning 5 epochs**, **50% weight pruning**, **static INT8 quantization**, **20% weight pruning** descendant 40개를 생성합니다. Adversarial fine-tuning은 실행하지 않습니다. 프로젝트와 결과를 Google Drive에 저장하므로 Colab runtime이 종료되어도 다음 세션에서 이어갈 수 있습니다.

In [ ]:
# 1. Google Drive를 연결합니다. 최초 실행 시 Google 계정 승인이 필요합니다.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Drive에 repository를 한 번만 clone합니다.
# 이미 존재하면 생성된 metadata/checkpoint를 보존한 채 최신 생성 코드를 가져옵니다.
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/chlwhdduq2357/Fingerprinting-Model-Zoo.git'
REPO = Path('/content/drive/MyDrive/Fingerprinting-Model-Zoo')
DATA_ROOT = Path('/content/drive/MyDrive/Fingerprinting-Model-Zoo-data')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('작업 경로:', Path.cwd())

In [ ]:
# 3. Colab에 이미 설치된 CUDA용 PyTorch는 유지하고 프로젝트만 editable install합니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)

# pip를 subprocess로 실행했으므로 현재 notebook kernel에 editable .pth를 즉시 반영합니다.
# 이 두 줄이 없으면 생성 CLI는 성공해도 현재 셀에서 `import model_zoo`가 실패할 수 있습니다.
import site
site.addsitedir(site.getsitepackages()[0])

import model_zoo
import torch
print('model_zoo:', model_zoo.__file__)
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('런타임 > 런타임 유형 변경에서 GPU를 선택한 뒤 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 4. C 생성에 필요한 A parent 10개만 선택적으로 다운로드합니다.
# 전체 A/B/B2 90개 archive를 받지 않습니다. 이미 hash가 맞는 파일은 자동으로 건너뜁니다.
PARENTS = ['A001', 'A002', 'A003', 'A006', 'A008', 'A010', 'A013', 'A015', 'A027', 'A029']
download_command = [
    sys.executable, 'scripts/download_models.py',
    '--model-id', *PARENTS, '--workers', '2'
]
subprocess.run(download_command, check=True)

In [ ]:
# 5. 먼저 작은 ResNet20(A002) 하나로 전체 pipeline을 확인합니다.
# 결과는 C005(FT5), C006(PRUNE50), C007(PTQ_INT8), C008(PRUNE20)입니다.
# 기존 C005/C006이 있으면 hash/config 확인 후 건너뛰고 C007/C008만 생성합니다.
pilot_command = [
    sys.executable, 'scripts/generate_c_fast.py',
    '--model-id', 'A002',
    '--transforms', 'ft5', 'prune50', 'ptq_int8', 'prune20',
    '--device', 'cuda', '--amp',
    '--data-root', str(DATA_ROOT),
    '--batch-size', '128', '--workers', '2'
]
subprocess.run(pilot_command, check=True)

In [ ]:
# 6. 나머지 parent를 한 모델씩 처리합니다.
# 세션이 끊기면 1~4번 셀을 다시 실행한 뒤 이 셀을 재실행하세요.
# checkpoint와 config hash가 맞는 완료 모델은 건너뛰므로 처음부터 다시 학습하지 않습니다.
for parent_id in PARENTS:
    print('\n' + '=' * 70)
    print('처리 중:', parent_id)
    command = [
        sys.executable, 'scripts/generate_c_fast.py',
        '--model-id', parent_id,
        '--transforms', 'ft5', 'prune50', 'ptq_int8', 'prune20',
        '--device', 'cuda', '--amp',
        '--data-root', str(DATA_ROOT),
        '--batch-size', '128', '--workers', '2'
    ]
    subprocess.run(command, check=True)

In [ ]:
# 7. 생성 수, 정확도, lineage를 확인합니다.
import json
rows = json.loads((REPO / 'metadata/models.json').read_text(encoding='utf-8'))
created = [row for row in rows if row.get('group') == 'C' and row.get('transform_type') in {'ft5', 'prune50', 'ptq_int8', 'prune20'}]
for row in sorted(created, key=lambda item: item['model_id']):
    print(
        row['model_id'], row['parent_id'], row['transform_type'],
        f"acc={row['cifar10_test_accuracy_verified']:.2f}%",
        'lineage=' + row['lineage_id']
    )
print(f'\n완료: {len(created)}/40')
assert len(created) == 40

In [ ]:
# 8. 기존 unified loader로 생성 모델을 실제 호출합니다.
from model_zoo import load_model, pair_relation
gpu_images = torch.rand(4, 3, 32, 32, device='cuda')  # normalize하지 않은 [0,1] RGB tensor
cpu_images = gpu_images.cpu()
ft_model = load_model('C005', device='cuda')       # A002 -> FT5
pruned50_model = load_model('C006', device='cuda') # A002 -> PRUNE50
quantized_model = load_model('C007', device='cpu') # A002 -> PTQ_INT8; quantized op는 CPU
pruned20_model = load_model('C008', device='cuda') # A002 -> PRUNE20
with torch.inference_mode():
    print('FT logits:', ft_model(gpu_images).shape)
    print('PRUNE50 logits:', pruned50_model(gpu_images).shape)
    print('PTQ_INT8 logits:', quantized_model(cpu_images).shape)
    print('PRUNE20 logits:', pruned20_model(gpu_images).shape)
print('A002-C005:', pair_relation('A002', 'C005'))
print('C005-C006:', pair_relation('C005', 'C006'))
print('C007-C008:', pair_relation('C007', 'C008'))

## 결과 위치

- Checkpoint: `checkpoints/C/`
- Epoch log: `runs/C/`
- Metadata: `metadata/models.json`, `metadata/models.csv`
- 마지막 실행 요약: `reports/c_fast_run.json`

Checkpoint는 `.gitignore` 대상입니다. 생성된 weight를 일반 Git commit에 추가하지 마세요. PTQ_INT8은 CUDA가 아니라 Colab CPU quantized backend에서 실행됩니다. A015 fine-tuning에서 CUDA OOM이 발생할 때만 5번/6번 셀의 `--batch-size 128`을 `64`로 바꾸고 해당 parent를 다시 실행하세요.